<a href="https://githubtocolab.com/geonextgis/smartreview/blob/main/docs/examples/example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

# SmartReview

This notebook walks through the recommended `smartreview` workflow:

1. Set the `OPENAI_API_KEY` environment variable (or use a `.env` file / Colab secrets).
2. Run the high-level `rank_papers` pipeline on a Web of Science export.
3. Inspect the ranked DataFrame and the files written to disk.
4. Optionally drop down to the lower-level API for custom analysis (similarity histograms, percentile thresholds, BibTeX, etc.).

It runs unchanged on **Google Colab** and on your **local machine** — the cells below auto-detect the environment.

---

## Running on Google Colab

1. Click the **Open in Colab** badge above (or upload this `.ipynb` to <https://colab.research.google.com/>).
2. **Add your OpenAI API key as a Colab secret:**
   - Click the **🔑 key icon** in the left sidebar.
   - Click **+ Add new secret**, name it `OPENAI_API_KEY`, paste your key as the value.
   - Toggle **Notebook access** on for that secret.
3. **Upload your Web of Science export** when prompted (cell §2). On Colab the file lives in `/content/` and is wiped when the runtime stops — re-upload on each session.
4. Run the cells top-to-bottom. The first cell installs `smartreview` from PyPI; everything else is identical to the local flow.

## Running locally

1. `pip install smartreview` (or `pip install -e .` from a clone).
2. Put your Web of Science export at `docs/examples/data/<your-file>.xls` and update `INPUT_FILE` below.
3. Provide your key via either an `OPENAI_API_KEY` env var or a `.env` file in the working directory.

## 1. Setup

In [ ]:
# Detect whether we're on Google Colab. The rest of the notebook branches on this.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
print('Running on Colab:', IN_COLAB)

In [ ]:
# On Colab the package isn't pre-installed, so install it on first run.
# Locally, you should `pip install -e .` from the repo root once instead.
if IN_COLAB:
    %pip install -q smartreview python-dotenv matplotlib

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from smartreview import (
    rank_papers,
    load_wos_export,
    create_openai_client,
    EmbeddingCache,
    embed_papers,
    get_embedding,
    calculate_cosine_similarity,
    get_top_k_papers,
    top_k_by_percentile,
    create_top_k_dataframe,
    save_top_k_papers,
    generate_bibtex_file,
    print_top_k_summary,
)

# --- API key handling -----------------------------------------------------
# Colab: read from Colab Secrets (the 🔑 sidebar). Local: read from a .env
# file or an existing OPENAI_API_KEY environment variable.
if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    except Exception as exc:
        raise RuntimeError(
            'Add a Colab secret named OPENAI_API_KEY: click the key icon in '
            'the left sidebar, then "+ Add new secret" and toggle Notebook '
            'access on.'
        ) from exc
else:
    from dotenv import load_dotenv
    load_dotenv()

if not os.environ.get('OPENAI_API_KEY'):
    raise RuntimeError('OPENAI_API_KEY is not set. See the setup section above.')
print('OPENAI_API_KEY set: True')

## 2. Provide your Web of Science export

On **Colab** the cell below opens an upload dialog (`/content/<filename>` is the resulting path). On **local** runs it expects the file to already exist on disk — set `LOCAL_INPUT_FILE` to its path.

In [ ]:
LOCAL_INPUT_FILE = 'data/test_data.xls'  # adjust to your file when running locally

if IN_COLAB:
    from google.colab import files
    print('Select your Web of Science .xls / .xlsx export to upload...')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded.')
    INPUT_FILE = next(iter(uploaded))  # first uploaded filename, lives in /content/
    OUTPUT_DIR = '/content/smartreview_out'
else:
    INPUT_FILE = LOCAL_INPUT_FILE
    OUTPUT_DIR = 'data'

print('Input file :', INPUT_FILE)
print('Output dir :', OUTPUT_DIR)

## 3. Define your research interests

In [ ]:
interest_text = """
I am interested in research at the intersection of machine learning, deep learning, and artificial intelligence applied to agriculture and crop yield prediction.
This includes methods such as LSTM, RNN, CNN, transformers, XGBoost, gradient boosting, and hybrid or process-based models like DSSAT or APSIM.
I am also interested in studies using remote sensing and Earth observation data, including Sentinel, Landsat, MODIS, NDVI, and EVI,
as well as climate change impacts such as drought, heat stress, and extreme events on crops.
Phenology, growing season analysis, crop calendars, and time series modeling are also relevant.
Overall, the focus is on improving food production systems, crop modeling, and global agricultural assessments.
""".strip()

## 4. `rank_papers`

`rank_papers` reads the export, embeds papers + interest, ranks by cosine similarity, and writes CSV / Excel / BibTeX to `OUTPUT_DIR`. Embeddings are cached on disk under `<OUTPUT_DIR>/embeddings/cache/`, so re-running this cell after editing your interest text will re-use paper embeddings and only re-embed the interest statement.

On Colab the cache lives under `/content/smartreview_out/embeddings/cache/` and disappears with the runtime — for long-running corpora, mount Google Drive (see §7) and point `OUTPUT_DIR` there to make the cache persistent.

In [ ]:
result = rank_papers(
    input_path=INPUT_FILE,
    interest_text=interest_text,
    output_dir=OUTPUT_DIR,
    top_percentile=80.0,  # keep the top 20%; use top_k=N for a fixed count
)

df = result['dataframe']
print(f"Selected {len(df)} papers.")
print(f"  CSV:    {result['files']['csv']}")
print(f"  Excel:  {result['files']['excel']}")
print(f"  BibTeX: {result['bibtex']['file']}")

In [ ]:
print_top_k_summary(df, k=len(df), show_rows=10)

In [ ]:
# On Colab, pull the generated files down to your local machine.
if IN_COLAB:
    from google.colab import files as colab_files
    for path in (result['files']['csv'], result['files']['excel'], result['bibtex']['file']):
        if path:
            colab_files.download(path)

## 5. Visualise the similarity distribution

In [ ]:
scores = np.array([s for _, s in result['similarities']])
threshold = result['top_k'][-1][1] if result['top_k'] else float('nan')

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(scores, bins=40, color='steelblue', edgecolor='black', alpha=0.75)
ax.axvline(threshold, color='red', linestyle='--', linewidth=2,
           label=f'Selection threshold: {threshold:.3f}')
ax.set_xlabel('Cosine similarity to interest statement')
ax.set_ylabel('Number of papers')
ax.set_title('Similarity score distribution')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Lower-level API (optional)

If you need finer control — e.g. custom text preprocessing, multiple interest queries against the same corpus, or a different embedding model — you can call the building blocks directly. The on-disk cache means you only pay the API once per (paper, model) pair.

In [ ]:
client = create_openai_client()
cache = EmbeddingCache(os.path.join(OUTPUT_DIR, 'embeddings', 'cache'))

data, summary = load_wos_export(INPUT_FILE)
paper_embeddings = embed_papers(summary, client=client, cache=cache)
interest_embedding = get_embedding(interest_text, client=client, cache=cache)

similarities = calculate_cosine_similarity(interest_embedding, paper_embeddings)

# Either a fixed K...
top_50 = get_top_k_papers(similarities, k=50)
# ...or a percentile-based threshold:
top_20pct = top_k_by_percentile(similarities, percentile=80.0)

top_df = create_top_k_dataframe(top_50, data)
save_top_k_papers(top_df, output_dir=OUTPUT_DIR, k=50)
generate_bibtex_file(top_df, output_dir=OUTPUT_DIR, k=50)
print(f'Wrote top 50 (fixed K) and selected {len(top_20pct)} papers via top 20%.')

## 7. (Colab only) Persist outputs to Google Drive

Colab runtimes are ephemeral — the embedding cache and exports vanish when the session stops. Mount your Drive once and point `OUTPUT_DIR` there to keep them across sessions:

```python
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/smartreview_out'  # then re-run §4
```

## 8. CLI alternative (local installs)

After `pip install smartreview` (or `pip install -e .`) the same pipeline is available as a shell command:

```bash
smartreview \
  --input data/papers.xls \
  --interest-file interest.txt \
  --output-dir data \
  --top-percentile 80
```